# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [1]:
from class_dataset import SegmentationClassDataset, num_classes as class_num_classes
from instance_dataset import SegmentationInstanceDataset

# mode = "instance"
mode = "class"

if mode == "instance":
    dataset_class = SegmentationInstanceDataset
    num_classes = 2
    model_path = "model_instance.pth"
elif mode == "class":
    dataset_class = SegmentationClassDataset
    num_classes = class_num_classes
    model_path = "model_class.pth"

In [2]:
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils

import torch


model = get_model_instance_segmentation(num_classes)

dataset = dataset_class("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

{'loss_classifier': tensor(0.7711, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.5219, grad_fn=<DivBackward0>), 'loss_mask': tensor(1.5862, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_objectness': tensor(1.5916, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.1896, grad_fn=<DivBackward0>)}
{'boxes': tensor([[2.2771e+02, 7.3264e+01, 3.8031e+02, 2.9715e+02],
        [1.4837e+02, 2.5099e+02, 3.9010e+02, 2.9822e+02],
        [1.5477e+02, 5.4954e+00, 3.6380e+02, 2.8504e+02],
        [5.6478e+01, 3.4682e+01, 2.3086e+02, 2.5313e+02],
        [3.8005e+02, 1.4214e+02, 4.0000e+02, 2.9388e+02],
        [3.6861e+02, 1.8846e+02, 3.9821e+02, 2.8722e+02],
        [3.5577e+02, 1.2851e+02, 4.0000e+02, 2.9716e+02],
        [4.1450e+01, 2.6319e+02, 2.7672e+02, 2.9889e+02],
        [2.1988e+02, 1.5095e+02, 3.4537e+02, 2.9515e+02],
        [3.7090e+02, 1.2283e+00, 4.0000e+02, 2.7714e+02],
        [3.1719e+02, 1.0498e+02, 3.9644e+02, 2.7295e+02],
        [2

## Training and Validation


In [3]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device("cpu")

# Check for accelerator availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA for training.")
else:
    print("Using CPU for training.")

Using CUDA for training.


In [4]:
from engine import evaluate, train_one_epoch


# use our dataset and defined transformations
dataset = dataset_class("../../training_data", get_transforms(is_train=True))
dataset_test = dataset_class("../../training_data", get_transforms(is_train=False))

# split the dataset in train and test set (last 30 images are test set)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-30])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-30:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn,
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_test, device=device)

print("That's it!")

# Save the model to disk
torch.save(model.state_dict(), model_path)

Epoch: [0]  [  0/120]  eta: 0:03:23  lr: 0.000047  loss: 4.6401 (4.6401)  loss_classifier: 0.6152 (0.6152)  loss_box_reg: 0.4801 (0.4801)  loss_mask: 1.4224 (1.4224)  loss_objectness: 1.8351 (1.8351)  loss_rpn_box_reg: 0.2872 (0.2872)  time: 1.6969  data: 0.4267  max mem: 1865
Epoch: [0]  [ 10/120]  eta: 0:02:14  lr: 0.000467  loss: 3.9147 (3.6189)  loss_classifier: 0.5440 (0.5275)  loss_box_reg: 0.5349 (0.4967)  loss_mask: 1.1794 (1.0482)  loss_objectness: 1.2380 (1.3425)  loss_rpn_box_reg: 0.2230 (0.2040)  time: 1.2236  data: 0.9165  max mem: 5053
Epoch: [0]  [ 20/120]  eta: 0:01:49  lr: 0.000886  loss: 2.4641 (2.6486)  loss_classifier: 0.3567 (0.4092)  loss_box_reg: 0.4864 (0.4512)  loss_mask: 0.4681 (0.7071)  loss_objectness: 0.7166 (0.8992)  loss_rpn_box_reg: 0.1650 (0.1820)  time: 1.0695  data: 0.8707  max mem: 5053
Epoch: [0]  [ 30/120]  eta: 0:01:32  lr: 0.001306  loss: 1.0678 (2.1249)  loss_classifier: 0.1478 (0.3166)  loss_box_reg: 0.3877 (0.4236)  loss_mask: 0.2943 (0.5635) 